In [ ]:
import numpy as np
import lightgbm as lgs
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, roc_curve, auc
from imblearn.over_sampling import SMOTE 
from geopy.distance import geodesic 
import joblib 

In [ ]:
df = pd.read_csv('dataset.csv')
df.head() 


In [ ]:
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])

df['hour'] = df['trans_date_trans_time'].dt.hour
df['day'] = df['trans_date_trans_time'].dt.day
df['month'] = df['trans_date_trans_time'].dt.month

In [ ]:
drop_columns=['Unnamed: 0','trans_date_trans_time','first','last','street','city', 'state','zip','dob','job','trans_num']
df = df.drop(columns=drop_columns)


In [ ]:
df.head()


In [58]:
# from sklearn.preprocessing import LabelEncoder
cat_col = ['merchant', 'category', 'gender']
encoders = {}

for col in cat_col:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

joblib.dump(encoders, "encoders_dict.jb")

['encoders_dict.jb']

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    return np.array([geodesic((a,b), (c,d)).km for a, b, c, d in zip( lat1,lon1,lat2,lon2)])
df['distance'] = haversine_distance (df['lat'],df['long'],df['merch_lat'], df['merch_long'])


In [ ]:
df.head()

In [ ]:
features = ['merchant','category','amt','cc_num','hour','day','month','gender','distance']
x = df[features]
y= df['is_fraud']
df.head()

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x='is_fraud', data=df,)
plt.title('Class Distribution before SMOTE')
plt.show() 

In [ ]:
smote = SMOTE(random_state=42)
x_resampled, y_resampled = smote.fit_resample(x, y)

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x=y_resampled)
plt.title('Class Distribution after SMOTE')
plt.show()

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x_resampled, y_resampled, test_size=0.2, random_state=42)

In [ ]:
# Ensure x_train and y_train are defined
if 'x_train' not in locals() or 'y_train' not in locals():
    from sklearn.model_selection import train_test_split
    x_train, x_test, y_train, y_test = train_test_split(x_resampled, y_resampled, test_size=0.2, random_state=42)

lgb_model = lgs.LGBMClassifier(
    boosting_type='gbdt',
    objective='binary',
    metrics='auc',
    is_unbalance=True,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    n_estimators=200,
)
lgb_model.fit(x_train, y_train)

In [ ]:
y_pred = lgb_model.predict(x_test)

In [ ]:
print("classification_report:\n", classification_report(y_test, y_pred))
print("roc_auc_score:", roc_auc_score(y_test, y_pred))

In [ ]:
lgs.plot_importance(lgb_model, max_num_features=10, importance_type='split', figsize=(10, 6))
plt.title('top 10 features by split importance')
plt.show()

In [ ]:
fpr , tpr, thresholds = roc_curve(y_test, lgb_model.predict_proba(x_test)[:, 1])
roc_auc = auc(fpr, tpr)

In [ ]:
plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='red', linestyle='--')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.show()

In [ ]:
joblib.dump(lgb_model, "fraud_detection_model.jb")
joblib.dump(encoders, "label_encoders.jb")

In [ ]:
import platform
print(platform.architecture())

In [ ]:
import joblib
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
encoder.fit(y_train)
joblib.dump(encoder, "label_encoder.jb")

